<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/kartik2006-del/efficient-finetuning-lora.git

fatal: destination path 'efficient-finetuning-lora' already exists and is not an empty directory.


In [2]:
%cd efficient-finetuning-lora

/content/efficient-finetuning-lora


In [3]:
!ls

lora_finetuning.ipynb  README.md


In [4]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [5]:
print("Number of examples:", len(dataset["train"]))
print("Columns:", dataset["train"].column_names)

print("\nFirst example:")
print(dataset["train"][0])

Number of examples: 51760
Columns: ['output', 'input', 'instruction']

First example:
{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'input': '', 'instruction': 'Give three tips for staying healthy.'}


In [6]:
for column in dataset["train"].column_names:
    missing = sum(
        1 for x in dataset["train"][column]
        if x is None or x == ""
    )
    print(column, ":", missing)

output : 0
input : 32603
instruction : 0


In [7]:
train_data = dataset["train"]

with_input = sum(1 for x in train_data["input"] if x.strip() != "")
without_input = len(train_data) - with_input

print("With input:", with_input)
print("Without input:", without_input)

With input: 19157
Without input: 32603


In [8]:
import random

for i in random.sample(range(len(train_data)), 5):
    print("=" * 80)
    print("Instruction:", train_data[i]["instruction"])
    print("Input:", train_data[i]["input"])
    print("Output:", train_data[i]["output"])

Instruction: Alex is at the candy store buying jellybeans. He wants to buy at least 100 jellybeans. He wants to buy the least amount such that he would have exactly 11 leftover after dividing the jellybeans evenly among 13 people. How many jellybeans should Alex buy?
Input: 
Output: To buy jellybeans, Alex needs to have at least 100 jellybeans, and wants to buy the minimum amount that would leave him with exactly 11 leftover after dividing the jellybeans evenly among 13 people. Since 100 divided by 13 leaves a remainder of 9, Alex needs 11 jellybeans left over, so he should buy 102 jellybeans (which is 2 more than 100).
Instruction: Provide a detailed explanation for why people are afraid of spiders.
Input: 
Output: Many people are afraid of spiders, a fear also known as arachnophobia. The reasons for why people develop this fear can vary, but there are several common explanations.

Firstly, one of the reasons why people are afraid of spiders is because of evolutionary biology. It is b

In [9]:
examples = [
    (row["instruction"], row["input"], row["output"])
    for row in train_data
]
unique_examples = len(set(examples))

print("Total examples:", len(examples))
print("Unique examples:", unique_examples)
print("Duplicate examples:", len(examples) - unique_examples)

Total examples: 51760
Unique examples: 51756
Duplicate examples: 4


In [10]:
instructions = train_data["instruction"]

unique_instructions = len(set(instructions))

print("Total instructions:", len(instructions))
print("Unique instructions:", unique_instructions)
print("Duplicate instructions:", len(instructions) - unique_instructions)

Total instructions: 51760
Unique instructions: 51725
Duplicate instructions: 35


In [11]:
instruction_lengths = [
    len(x.split()) for x in train_data["instruction"]
]

output_lengths = [
    len(x.split()) for x in train_data["output"]
]

print("Instruction length:")
print("Minimum:", min(instruction_lengths))
print("Maximum:", max(instruction_lengths))
print("Average:", sum(instruction_lengths) / len(instruction_lengths))

print("\nOutput length:")
print("Minimum:", min(output_lengths))
print("Maximum:", max(output_lengths))
print("Average:", sum(output_lengths) / len(output_lengths))

Instruction length:
Minimum: 2
Maximum: 323
Average: 10.542716383307573

Output length:
Minimum: 1
Maximum: 507
Average: 109.9395285935085


In [12]:
max_instruction_index = instruction_lengths.index(max(instruction_lengths))

print("Longest instruction:")
print(train_data[max_instruction_index]["instruction"])

print("\nWord count:", instruction_lengths[max_instruction_index])

Longest instruction:
You are a smart assistant designed to help high school teachers come up with reading comprehension questions.
Given a piece of text, you must come up with a question and answer pair that can be used to test a student's reading comprehension abilities.
When coming up with this question/answer pair, you must respond in the following format:
```
{{
    "question": "$YOUR_QUESTION_HERE",
    "answer": "$THE_ANSWER_HERE"
}}
```
Everything between the ``` must be valid json.
Please come up with a question/answer pair, in the specified JSON format, for the following text:
----------------
Previously, the use of dye-sub printing was limited to industrial or high-end commercial printing. Dye-sub photo printing has been used in medical imaging, graphic arts proofing, security, and broadcast-related applications. Today, it is extremely popular in event photography and photo booths or kiosks that require high-speed, on-demand printing.

Alps Electric produced the first quality

In [13]:
max_output_index = output_lengths.index(max(output_lengths))

print("Longest output:")
print(train_data[max_output_index]["output"])

print("\nWord count:", output_lengths[max_output_index])

Longest output:
Here is an implementation of a red-black tree in C++:

```
#include <iostream>

using namespace std;

enum Color {RED, BLACK};

struct Node {
    int val;
    Color color;
    Node *left, *right, *parent;
    
    Node() {}
    
    Node(int v, Color c) {
        val = v;
        color = c;
        left = right = parent = NULL;
    }
};

class RedBlackTree {
    private:
        Node *root;
        
    public:
        RedBlackTree() { root = NULL; }
        
        /* Helper functions */
        Node* grandparent(Node *n) {
            if (n && n->parent) { return n->parent->parent; }
            else { return NULL; }
        }
        
        Node* uncle(Node *n) {
            Node *g = grandparent(n);
            if (!g) { return NULL; }
            if (n->parent == g->left) { return g->right; }
            else { return g->left; }
        }
        
        Node* sibling(Node *n) {
            if (!n || !n->parent) { return NULL; }
            if (n == n->parent->

In [14]:
df = train_data.to_pandas()

print("Before removing duplicates:", len(df))

df = df.drop_duplicates(
    subset=["instruction", "input", "output"]).reset_index(drop=True)

print("After removing duplicates:", len(df))
print("Removed:", 51760 - len(df))

Before removing duplicates: 51760
After removing duplicates: 51756
Removed: 4


In [15]:
def format_example(row):
    if row["input"].strip():
        user_content = row["instruction"] + "\n\nInput:\n" + row["input"]
    else:
        user_content = row["instruction"]

    return {
        "messages": [
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": row["output"]
            }
        ]
    }

In [16]:
from datasets import Dataset

formatted_data = Dataset.from_pandas(df)

formatted_data = formatted_data.map(
    format_example,
    remove_columns=["instruction", "input", "output"]
)

Map:   0%|          | 0/51756 [00:00<?, ? examples/s]

In [17]:
print(formatted_data[0])

{'messages': [{'content': 'Give three tips for staying healthy.', 'role': 'user'}, {'content': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.', 'role': 'assistant'}]}


In [18]:
train_test = formatted_data.train_test_split(
    test_size=0.20,
    seed=42
)

train_data = train_test["train"]

temp_data = train_test["test"].train_test_split(
    test_size=0.50,
    seed=42
)

validation_data = temp_data["train"]
test_data = temp_data["test"]

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Train: 41404
Validation: 5176
Test: 5176


In [19]:
print(train_data[0])
print("\nTrain:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))
print(train_data.column_names)

{'messages': [{'content': 'Imagine you are advising a fictional character who seeks to purchase a suitable spaceship for interstellar travel. Recommend three critical factors they should consider and explain why each is important.', 'role': 'user'}, {'content': "1. Fuel efficiency - To maximize reach and minimize fuel costs, it's essential to choose a ship with excellent fuel efficiency.\\n2. Advanced navigation system - Accurate navigation is crucial for avoiding hazards and reaching intended destinations in the vastness of space.\\n3. Maintenance and repair capabilities - The ship should be easy to maintain and repair, as it may become necessary during long journeys with limited access to external services.", 'role': 'assistant'}]}

Train: 41404
Validation: 5176
Test: 5176
['messages']


In [20]:
!pip install -q transformers accelerate

In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [22]:
prompt = "Explain machine learning in simple words."

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True)

print(response)

Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance over time without being explicitly programmed. It involves using algorithms to analyze patterns in large amounts of data and make predictions or decisions based on those patterns.

In simpler terms, imagine you have a bunch of toys scattered around your room. Machine learning can help you organize these toys into different categories, find the ones you like best, and even suggest new toys for you to play with if you're not sure which one


In [23]:
!pip install -q peft

In [24]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM")

In [25]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

GPU available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [26]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False
        )
    }

train_text = train_data.map(format_chat)
validation_text = validation_data.map(format_chat)

Map:   0%|          | 0/41404 [00:00<?, ? examples/s]

Map:   0%|          | 0/5176 [00:00<?, ? examples/s]

In [27]:
print(train_text[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Imagine you are advising a fictional character who seeks to purchase a suitable spaceship for interstellar travel. Recommend three critical factors they should consider and explain why each is important.<|im_end|>
<|im_start|>assistant
1. Fuel efficiency - To maximize reach and minimize fuel costs, it's essential to choose a ship with excellent fuel efficiency.\n2. Advanced navigation system - Accurate navigation is crucial for avoiding hazards and reaching intended destinations in the vastness of space.\n3. Maintenance and repair capabilities - The ship should be easy to maintain and repair, as it may become necessary during long journeys with limited access to external services.<|im_end|>



In [28]:
tokenizer.pad_token = tokenizer.eos_token


In [29]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

In [30]:
tokenized_train = train_text.map(
    tokenize_function,
    batched=True,
    remove_columns=train_text.column_names
)
tokenized_validation = validation_text.map(
    tokenize_function,
    batched=True,
    remove_columns=validation_text.column_names
)

Map:   0%|          | 0/41404 [00:00<?, ? examples/s]

Map:   0%|          | 0/5176 [00:00<?, ? examples/s]

In [31]:
print(tokenized_train[0])

{'input_ids': [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 51057, 498, 525, 62679, 264, 43582, 3668, 879, 25654, 311, 7627, 264, 14452, 84842, 369, 946, 77293, 5821, 13, 46606, 2326, 9023, 9363, 807, 1265, 2908, 323, 10339, 3170, 1817, 374, 2989, 13, 151645, 198, 151644, 77091, 198, 16, 13, 36284, 15024, 481, 2014, 34508, 5545, 323, 29337, 10416, 7049, 11, 432, 594, 7565, 311, 5157, 264, 8284, 448, 9073, 10416, 15024, 7110, 77, 17, 13, 21159, 10646, 1849, 481, 11429, 61159, 10646, 374, 16587, 369, 30426, 51542, 323, 18739, 10602, 33105, 304, 279, 12767, 2090, 315, 3550, 7110, 77, 18, 13, 38206, 323, 12733, 16928, 481, 576, 8284, 1265, 387, 4135, 311, 10306, 323, 12733, 11, 438, 432, 1231, 3635, 5871, 2337, 1293, 63776, 448, 7199, 2615, 311, 9250, 3516, 13, 151645, 198, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151645, 151

In [37]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [39]:
import torch
import peft

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)

PyTorch: 2.11.0+cu128
PEFT: 0.20.0


In [40]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [41]:
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [42]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.2.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.3.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.3.self_attn.q_proj.lora_B.default.weight
base_m

In [43]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs/lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)

In [45]:
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [46]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [47]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator
)

In [44]:
TrainingArguments(
    output_dir="outputs/lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_star

In [50]:
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

In [51]:
print("Global step:", trainer.state.global_step)
print("Epoch:", trainer.state.epoch)
print("Last logs:", trainer.state.log_history[-5:])

Global step: 1
Epoch: 0.00038643609313109847
Last logs: []


In [52]:
!ls -lah ./outputs/lora

total 12K
drwxr-xr-x 3 root root 4.0K Sep 26 11:01 .
drwxr-xr-x 3 root root 4.0K Sep 26 09:46 ..
drwxr-xr-x 2 root root 4.0K Sep 26 11:01 checkpoint-2588


In [53]:
!ls -lah ./outputs/lora/checkpoint-2588

total 24M
drwxr-xr-x 2 root root 4.0K Sep 26 11:01 .
drwxr-xr-x 3 root root 4.0K Sep 26 11:01 ..
-rw-r--r-- 1 root root 1.1K Sep 26 11:01 adapter_config.json
-rw------- 1 root root 4.2M Sep 26 11:01 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K Sep 26 11:01 chat_template.jinja
-rw-r--r-- 1 root root 8.4M Sep 26 11:01 optimizer.pt
-rw-r--r-- 1 root root 5.1K Sep 26 11:01 README.md
-rw-r--r-- 1 root root  15K Sep 26 11:01 rng_state.pth
-rw-r--r-- 1 root root 1.4K Sep 26 11:01 scaler.pt
-rw-r--r-- 1 root root 1.5K Sep 26 11:01 scheduler.pt
-rw-r--r-- 1 root root  691 Sep 26 11:01 tokenizer_config.json
-rw-r--r-- 1 root root  11M Sep 26 11:01 tokenizer.json
-rw-r--r-- 1 root root  48K Sep 26 11:01 trainer_state.json
-rw-r--r-- 1 root root 5.1K Sep 26 11:01 training_args.bin


In [54]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
